 # Manually recalculating/refitting AVAR

In [1]:
import gc
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings;

warnings.filterwarnings('ignore')

In [2]:
# DATA_PATH = 'updated-data/obs/lme-ready'
DATA_PATH = '/Volumes/Data-Repo/shapeoflang3/obs'
NULL_PATH = '/Volumes/Data-Repo/shapeoflang3/null-files/null'

In [3]:
fs = [f for f in os.listdir(DATA_PATH) if f.endswith('.parquet') and (not f.startswith('.'))]
len(fs), fs[:3]

(5591,
 ['CABNC-CABNC-KB0RE006-0missing.parquet',
  'CABNC-CABNC-KB0RE007-KB0.parquet',
  'CABNC-CABNC-KB0RE001-KB0.parquet'])

In [4]:
fn = [f for f in os.listdir(NULL_PATH) if f.endswith('.parquet') and (not f.startswith('.'))]
len(fn), fn[:3]

(5595,
 ['CORAAL-CORAAL-se2_ag1_m_02DCA-se2-ag1.parquet',
  'xling-finchat-fin-finchat-fin-7.parquet',
  'CABNC-CABNC-KB0RE00H-KB0.parquet'])

In [5]:
val_col = 'I'
turn_col = 'x_turn_id'
conversation_id_col = 'conversation_id'

# origin/correct
# merge_cols = [conversation_id_col, turn_col, 'self']

##### Experiments
merge_cols = [turn_col, 'y_speaker']
# merge_cols = [conversation_id_col, turn_col]

In [6]:
# safety measure... the columns in the origina raw $I$ data.
orig_columns = [
    'index',
    'x_speaker',
    'y_speaker',
    'x_turn_id',
    'y_turn_id',
    'conversation_id',
    'corpus',
    'nx',
    'ny',
    'CoS',
    'aCoS',
    'I',
    'H'
]

## (1) Allan Factor

Still a work in progress.

In [7]:
AVAR_PATH = '/'.join(DATA_PATH.split('/')[:-1])
AVAR_PATH = os.path.join(AVAR_PATH, 'AF')
if not os.path.exists(AVAR_PATH):
    os.mkdir(AVAR_PATH)

In [8]:
# base 2 growth
m = np.array([2,4,16,32,64,128])

# base 10 growth
# m = np.array([1,10,100,1000])

##### Experimental AF function?

In [9]:
def calc_af(x, mj, stride=1):
    """  Main algorithm for adev() (stride=mj) and oadev() (stride=1)

    see:
    https://https://github.com/aewallin/allantools/blob/master/allantools/allantools.py
    Parameters
    ----------
    phase: np.array
        Phase data in seconds.
    rate: float
        The sampling rate for phase or frequency, in Hz
    mj: int
        averaging factor, we evaluate at tau = m*tau0
    stride: int
        Size of stride

    Returns
    -------
    (dev, deverr, n): tuple
        Array of computed values.

    Notes
    -----
    stride = mj for nonoverlapping Allan deviation
    stride = 1 for overlapping Allan deviation

    References
    ----------
    * http://en.wikipedia.org/wiki/Allan_variance
    * http://www.leapsecond.com/tools/adev_lib.c
    * NIST [SP1065]_ eqn (7) and (11) page 16
    """
    x_ = np.cumsum(x)

    mj = int(mj)
    stride = int(stride)
    d1 = x_[1 * mj::stride]
    d0 = x_[::stride]

    n = min(len(d0), len(d1))

    if n == 0:
        # RuntimeWarning("Data array length is too small: %i" % len(phase))
        n = 1

    AF = ((d1[:n] - d0[:n])**2) / (2*d0[:n])
    AF = AF.sum() / n

    return AF, n

def integral(y,t):
    y_ = (y[:-1] + y[1:])/2
    t_ = t[1:] - t[:-1]
    return np.cumsum(y_ * t_)

### (1.1) Observed data

In [9]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'obs')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fs):
    f_ = os.path.join(DATA_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[orig_columns]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    # df['EXP'] = groups[val_col]
    df['min_y_turn_id'] = groups['y_turn_id'].transform('min')
    df['mtau'] = df['y_turn_id'] - df['min_y_turn_id']
    df['mu'] = groups[val_col].transform('mean')
    df['n'] = groups.cumcount()+1
    df['obs'] = (df[val_col] - df['mu'] + 1e-13)**2
    # df['obs'] = groups['obs'].cumsum()
    df['AF'] = df['obs'] / (2*df['mu'])
    df['AF'] = groups['AF'].cumsum()

    df[
        ['conversation_id', 'x_speaker', 'y_speaker', 'x_turn_id', 'y_turn_id', 'mtau', 'mu', 'n', 'AF']
    ].to_parquet(
        os.path.join(output_path,f),
        compression='snappy',
        engine='fastparquet'
    )
    #############################################################

  0%|          | 0/5591 [00:00<?, ?it/s]

In [ ]:
list(groups)[0][-1].head(n=20)

In [ ]:
df[['x_turn_id', 'y_speaker', 'y_turn_id', 'mtau', 'AF']].sort_values(by=['x_turn_id', 'y_speaker']).head(n=20)

### (1.2) Null data

In [ ]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'null')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fn):
    f_ = os.path.join(NULL_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[[col for col in orig_columns if col in list(df)]]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['y_turn_id'] = df.groupby(by=['x_turn_id', 'y_speaker']).cumcount() + 1
    # df['y_turn_id'] += df['x_turn_id']

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    for (xid, spy), dfi in groups:
        spx = dfi['x_speaker'].unique()[0]
        cid = dfi['conversation_id'].unique()[0]
        x = dfi['I'].values
        # m = np.logspace(np.log(1), np.log(100), num=100, base=2)
        m_ = m[m<len(x)]
        for mj in m_:
            avar, n = calc_avar(x,mj)
            outputs += [
                {
                    'conversation_id': cid,
                    'x_speaker': spx,
                    'y_speaker': spy,
                    'x_turn_id': xid,
                    'n': n,
                    'mtau': mj,
                    'avar': avar
                }
            ]
    if len(outputs):
        outputs = pd.DataFrame(outputs)
        outputs.to_parquet(
            os.path.join(output_path, f),
            engine='fastparquet',
            compression='snappy'
        )
    #############################################################

## (2) Continuously sampled AVAR

_run time: 17m_

In [ ]:
AVAR_PATH = '/'.join(DATA_PATH.split('/')[:-1])
AVAR_PATH = os.path.join(AVAR_PATH, 'AVAR')
if not os.path.exists(AVAR_PATH):
    os.mkdir(AVAR_PATH)

In [ ]:
def ___avar(x, k_clusters, data_max_dt, min_Dt=1, max_Dt=200):
    result = np.logspace(np.log(min_Dt), np.log(max_Dt), num=k_clusters, base=10)

    cluster_sizes = np.unique(np.round(result)).astype(int)
    cluster_sizes = cluster_sizes[cluster_sizes <= data_max_dt]

    x_ = np.cumsum(x)
    avar = np.empty(shape=cluster_sizes.shape + x_.shape[1:])
    for i,k in enumerate(cluster_sizes):
        avar[i] = ((x_[2*k:] - x_[k:-k] + x_[:-2*k])**2).mean() / (k**2)

    return cluster_sizes, avar

def calc_avar(x, mj, stride=1):
    """  Main algorithm for adev() (stride=mj) and oadev() (stride=1)

    see:
    https://https://github.com/aewallin/allantools/blob/master/allantools/allantools.py
    Parameters
    ----------
    phase: np.array
        Phase data in seconds.
    rate: float
        The sampling rate for phase or frequency, in Hz
    mj: int
        averaging factor, we evaluate at tau = m*tau0
    stride: int
        Size of stride

    Returns
    -------
    (dev, deverr, n): tuple
        Array of computed values.

    Notes
    -----
    stride = mj for nonoverlapping Allan deviation
    stride = 1 for overlapping Allan deviation

    References
    ----------
    * http://en.wikipedia.org/wiki/Allan_variance
    * http://www.leapsecond.com/tools/adev_lib.c
    * NIST [SP1065]_ eqn (7) and (11) page 16
    """
    mj = int(mj)
    stride = int(stride)
    d2 = x[2 * mj::stride]
    d1 = x[1 * mj::stride]
    d0 = x[::stride]

    n = min(len(d0), len(d1), len(d2))

    if n == 0:
        # RuntimeWarning("Data array length is too small: %i" % len(phase))
        n = 1

    v_arr = d2[:n] - (2 * d1[:n]) + d0[:n]
    s = (v_arr**2).sum()

    # not a biased estimator? n is not (n-1)?
    avar = s / (2.0*n*(mj**2))

    return avar, n

def calc_mavar(x, t, mj, stride=1):
    """  Main algorithm for adev() (stride=mj) and oadev() (stride=1)

    see:
    https://https://github.com/aewallin/allantools/blob/master/allantools/allantools.py
    Parameters
    ----------
    phase: np.array
        Phase data in seconds.
    rate: float
        The sampling rate for phase or frequency, in Hz
    mj: int
        averaging factor, we evaluate at tau = m*tau0
    stride: int
        Size of stride

    Returns
    -------
    (dev, deverr, n): tuple
        Array of computed values.

    Notes
    -----
    stride = mj for nonoverlapping Allan deviation
    stride = 1 for overlapping Allan deviation

    References
    ----------
    * http://en.wikipedia.org/wiki/Allan_variance
    * http://www.leapsecond.com/tools/adev_lib.c
    * NIST [SP1065]_ eqn (7) and (11) page 16
    """
    mj = int(mj)
    stride = int(stride)
    d2 = x[2 * mj::stride]
    d1 = x[1 * mj::stride]
    d0 = x[::stride]

    t2 = t[2 * mj::stride]
    t0 = t[::stride]

    n = min(len(d0), len(d1), len(d2))

    if n == 0:
        # RuntimeWarning("Data array length is too small: %i" % len(phase))
        n = 1

    tsel = (t2[:n] - t0[:n]) >= mj

    v_arr = d2[:n] - (2 * d1[:n]) + d0[:n]

    s = (v_arr[tsel]**2).sum()

    # not a biased estimator? n is not (n-1)?
    avar = s / (2.0*tsel.sum()*(mj**2))

    return avar, tsel.sum()

In [ ]:
# base 2 growth
m = np.array([2,4,16,32,64,128])

# base 10 growth
# m = np.array([1,10,100,1000])

### (2.1) Observed data

In [ ]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'obs')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fs):
    f_ = os.path.join(DATA_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[orig_columns]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    for (xid, spy), dfi in groups:
        spx = dfi['x_speaker'].unique()[0]
        cid = dfi['conversation_id'].unique()[0]

        # raw data
        # x = dfi['I'].values

        # phase-transformed data... This is the moment-to-moment integral.
        x = np.cumsum(dfi['I'].values - dfi['I'].mean())

        # m = np.logspace(np.log(1), np.log(100), num=100, base=2)
        m_ = m[m<len(x)]
        for mj in m_:
            avar, n = calc_avar(x,mj)
            outputs += [
                {
                    'conversation_id': cid,
                    'x_speaker': spx,
                    'y_speaker': spy,
                    'x_turn_id': xid,
                    'n': n,
                    'mtau': mj,
                    'avar': avar
                }
            ]

    if len(outputs):
        outputs = pd.DataFrame(outputs)
        outputs.to_parquet(
            os.path.join(output_path, f),
            engine='fastparquet',
            compression='snappy'
        )
    #############################################################

##### Last check

In [ ]:
import pyarrow.parquet as pq
bad_pqs = []
for f in tqdm(fs):
    try:
        table = pq.ParquetFile(os.path.join(output_path, f))
        if table.schema.names != ['conversation_id', 'x_speaker', 'y_speaker', 'x_turn_id', 'n', 'mtau', 'avar']:
            bad_pqs += [f]
    except Exception:
        0
len(bad_pqs)

In [ ]:
bad_pqs

### (2.2) Null data

In [ ]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'null')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fn):
    f_ = os.path.join(NULL_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[[col for col in orig_columns if col in list(df)]]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['y_turn_id'] = df.groupby(by=['x_turn_id', 'y_speaker']).cumcount() + 1
    # df['y_turn_id'] += df['x_turn_id']

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    for (xid, spy), dfi in groups:
        spx = dfi['x_speaker'].unique()[0]
        cid = dfi['conversation_id'].unique()[0]

        # raw data
        # x = dfi['I'].values

        # phase-transformed data... This is the moment-to-moment integral.
        x = np.cumsum(dfi['I'].values - dfi['I'].mean())

        # m = np.logspace(np.log(1), np.log(100), num=100, base=2)
        m_ = m[m<len(x)]
        for mj in m_:
            avar, n = calc_avar(x,mj)
            outputs += [
                {
                    'conversation_id': cid,
                    'x_speaker': spx,
                    'y_speaker': spy,
                    'x_turn_id': xid,
                    'n': n,
                    'mtau': mj,
                    'avar': avar
                }
            ]
    if len(outputs):
        outputs = pd.DataFrame(outputs)
        outputs.to_parquet(
            os.path.join(output_path, f),
            engine='fastparquet',
            compression='snappy'
        )
    #############################################################

## **_[CURRENT]_** (3) Exponential growth/accretion of information

The basic premise of this hypothesis is that information accumulates in a fractal pattern relative to any one starting point. It's not a measurement of frequency in this case, but power-law scaling in accretion patterns.

To measure accretion over time, we use the cumulative density function as defined as follows:

$$CDF(\tau) = \int_{i>1}^{\tau} I(x;y_\tau) \partial (\tau)$$

Our hypothesis/model is defined further as:

$$CDF(\tau) = e^{\beta_0}\tau^{\alpha}$$

Where $\tau$ is equal to $\Delta_t$ in other formulations, and is defined as:

$$\left[ \min t(y)_{sp(y)} > t(x) \right] - \left[t(y)_{sp(y)} > t(x) \right]_{y \in Y_{sp(y)}} + 1$$

In [7]:
AVAR_PATH = '/'.join(DATA_PATH.split('/')[:-1])
AVAR_PATH = os.path.join(AVAR_PATH, 'INT')
if not os.path.exists(AVAR_PATH):
    os.mkdir(AVAR_PATH)

In [8]:
# base 2 growth
m = np.array([1,2,4,16,32,64,128,200])

# base 10 growth
# m = np.array([1,10,100,1000])

In [9]:
def integral(y,t):
    y_ = (y[:-1] + y[1:])/2
    t_ = t[1:] - t[:-1]
    return np.cumsum(y_ * t_)

### (3.1) Observed data

_run time 23m_

In [10]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'obs')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fs):
    f_ = os.path.join(DATA_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[orig_columns]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    # df['EXP'] = groups[val_col]
    df['min_y_turn_id'] = groups['y_turn_id'].transform('min')
    df['mtau'] = df['y_turn_id'] - df['min_y_turn_id'] + 1

    for (xid, spy), dfi in groups:
        if len(dfi) > 1:
            spx = dfi['x_speaker'].unique()[0]
            cid = dfi['conversation_id'].unique()[0]
            nx = dfi['nx'].unique()[0]
            ymin = dfi['min_y_turn_id'].unique()[0]

            mtau = dfi['mtau'].values
            t = np.linspace(0,len(dfi)-1,len(dfi)).astype(int)

            integ = integral(dfi['I'].values, t)
            integ = integ / integ[-1]

            t = t[:-1]
            mtau = mtau[:-1]
            m_ = m[
                (m <= max(mtau))
            ]


            for mj in m_:
                n = t[mtau<=mj][-1]
                # print(xid, spy.split('-')[-1], mj, mtau[n], integ[n].item())
                # print()
                outputs += [
                    {
                        'conversation_id': cid,
                        'x_speaker': spx,
                        'y_speaker': spy,
                        'x_turn_id': xid,
                        'min_y_turn_id':ymin,
                        'nx': nx,
                        'n': n+1,
                        'mj': mj,
                        'mtau': mtau[mtau<=mj][-1],
                        'avar': integ[n]
                    }
                ]

    if len(outputs):
        outputs = pd.DataFrame(outputs)
        outputs = outputs.drop_duplicates(subset=[col for col in list(outputs) if col not in ['mj', 'mtau']])
        outputs.index = range(len(outputs))
        outputs.to_parquet(
            os.path.join(output_path,f),
            compression='snappy',
            engine='fastparquet'
        )
    #############################################################

  0%|          | 0/5591 [00:00<?, ?it/s]

##### Last checks

In [11]:
outputs.head(n=20)

,conversation_id,x_speaker,y_speaker,x_turn_id,min_y_turn_id,nx,n,mj,mtau,avar
0,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-INV,1,33,55,1,1,1,0.248146
1,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-INV,1,33,55,2,16,5,0.491254
2,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-INV,1,33,55,3,32,22,0.755725
3,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-PAR,1,2,55,1,1,1,0.008557
4,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-PAR,1,2,55,2,2,2,0.021441
5,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-PAR,1,2,55,4,4,4,0.040425
6,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-PAR,1,2,55,16,16,16,0.134667
7,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-PAR,1,2,55,31,32,31,0.265812
8,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-PAR,1,2,55,61,64,64,0.518801
9,yiddish-9,yiddish-yiddish-9-PAR,yiddish-yiddish-9-INV,2,33,35,1,1,1,0.257679


In [35]:
mj, mtau, integ

(np.int64(1), array([1]), array([0.2377903]))

### (3.2) Null data

In [10]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'null')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fn):
    f_ = os.path.join(NULL_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[orig_columns]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['y_turn_id'] = df.groupby(by=merge_cols).cumcount() + 1
    # df['y_turn_id'] = df['x_turn_id'] + df['y_turn_id']

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    # df['EXP'] = groups[val_col]
    df['min_y_turn_id'] = groups['y_turn_id'].transform('min')
    df['mtau'] = df['y_turn_id'] - df['min_y_turn_id'] + 1

    for (xid, spy), dfi in groups:
        if len(dfi) > 1:
            spx = dfi['x_speaker'].unique()[0]
            cid = dfi['conversation_id'].unique()[0]
            nx = dfi['nx'].unique()[0]
            ymin = dfi['min_y_turn_id'].unique()[0]

            mtau = dfi['mtau'].values
            t = np.linspace(0,len(dfi)-1,len(dfi)).astype(int)

            integ = integral(dfi['I'].values, t)
            integ = integ / integ[-1]

            t = t[:-1]
            mtau = mtau[:-1]
            m_ = m[
                (m <= max(mtau))
            ]


            for mj in m_:
                n = t[mtau<=mj][-1]
                # print(xid, spy.split('-')[-1], mj, mtau[n], integ[n].item())
                # print()
                outputs += [
                    {
                        'conversation_id': cid,
                        'x_speaker': spx,
                        'y_speaker': spy,
                        'x_turn_id': xid,
                        'min_y_turn_id':ymin,
                        'nx': nx,
                        'n': n+1,
                        'mj': mj,
                        'mtau': mtau[mtau<=mj][-1],
                        'avar': integ[n]
                    }
                ]

    if len(outputs):
        outputs = pd.DataFrame(outputs)
        outputs = outputs.drop_duplicates(subset=[col for col in list(outputs) if col not in ['mj', 'mtau']])
        outputs.index = range(len(outputs))
        outputs.to_parquet(
            os.path.join(output_path,f),
            compression='snappy',
            engine='fastparquet'
        )
    #############################################################

  0%|          | 0/5595 [00:00<?, ?it/s]

## (4) Exponential growth in integral for second-order differences (AVAR)

In [7]:
AVAR_PATH = '/'.join(DATA_PATH.split('/')[:-1])
AVAR_PATH = os.path.join(AVAR_PATH, 'INT')
if not os.path.exists(AVAR_PATH):
    os.mkdir(AVAR_PATH)

In [8]:
# base 2 growth
m = np.array([1,2,4,16,32,64,128,200])

# base 10 growth
# m = np.array([1,10,100,1000])

In [9]:
def calc_avar(x, mj=1, stride=1):
    """  Main algorithm for adev() (stride=mj) and oadev() (stride=1)

    see:
    https://https://github.com/aewallin/allantools/blob/master/allantools/allantools.py
    Parameters
    ----------
    phase: np.array
        Phase data in seconds.
    rate: float
        The sampling rate for phase or frequency, in Hz
    mj: int
        averaging factor, we evaluate at tau = m*tau0
    stride: int
        Size of stride

    Returns
    -------
    (dev, deverr, n): tuple
        Array of computed values.

    Notes
    -----
    stride = mj for nonoverlapping Allan deviation
    stride = 1 for overlapping Allan deviation

    References
    ----------
    * http://en.wikipedia.org/wiki/Allan_variance
    * http://www.leapsecond.com/tools/adev_lib.c
    * NIST [SP1065]_ eqn (7) and (11) page 16
    """
    mj = int(mj)
    stride = int(stride)
    d2 = x[2 * mj::stride]
    d1 = x[1 * mj::stride]
    d0 = x[::stride]

    n = min(len(d0), len(d1), len(d2))

    if n == 0:
        # RuntimeWarning("Data array length is too small: %i" % len(phase))
        n = 1

    v_arr = d2[:n] - (2 * d1[:n]) + d0[:n]
    s = (v_arr**2)

    # not a biased estimator? n is not (n-1)?
    avar = s / (2.0*(n**2)*mj)

    return np.cumsum(avar), n

### (4.1) Observed data

In [ ]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'obs')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fs):
    f_ = os.path.join(DATA_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[orig_columns]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    # df['EXP'] = groups[val_col]
    df['min_y_turn_id'] = groups['y_turn_id'].transform('min')
    df['mtau'] = df['y_turn_id'] - df['min_y_turn_id'] + 1

    for (xid, spy), dfi in groups:
        if len(dfi) > 2:
            spx = dfi['x_speaker'].unique()[0]
            cid = dfi['conversation_id'].unique()[0]
            nx = dfi['nx'].unique()[0]
            ymin = dfi['min_y_turn_id'].unique()[0]

            mtau = dfi['mtau'].values
            t = np.linspace(0,len(dfi)-1,len(dfi)).astype(int)

            integ,n = calc_avar(dfi['I'].values)
            integ = integ / integ[-1]

            t = t[:n]
            mtau = mtau[:n]
            m_ = m[
                (m <= max(mtau))
            ]


            for mj in m_:
                n = t[mtau<=mj][-1]
                # print(xid, spy.split('-')[-1], mj, mtau[n], integ[n].item())
                # print()
                outputs += [
                    {
                        'conversation_id': cid,
                        'x_speaker': spx,
                        'y_speaker': spy,
                        'x_turn_id': xid,
                        'min_y_turn_id':ymin,
                        'nx': nx,
                        'n': n+1,
                        'mj': mj,
                        'mtau': mtau[mtau<=mj][-1],
                        'avar': integ[n]
                    }
                ]

    if len(outputs):
        outputs = pd.DataFrame(outputs)
        outputs = outputs.drop_duplicates(subset=[col for col in list(outputs) if col not in ['mj', 'mtau']])
        outputs.index = range(len(outputs))
        outputs.to_parquet(
            os.path.join(output_path,f),
            compression='snappy',
            engine='fastparquet'
        )
    #############################################################